# Recipe Complexity and Ratings: What Makes a Recipe Successful?

**Name(s)**: Uddhav Panchavati, Ryan Namdar

**Website Link**: https://paymentblocked.github.io/recpies_and_interactions/



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import *

## Step 1: Introduction

### Dataset: Recipes and Ratings

This project uses the **Recipes and Ratings** dataset scraped from Food.com. It contains two files:

- **`RAW_recipes.csv`** — ~83,000 unique recipes with ingredients, steps, nutritional info, and tags
- **`RAW_interactions.csv`** — ~731,000 user reviews with 1-5 star ratings and text reviews

---

### Central Question

> **Does recipe complexity -- measured by number of steps and ingredients -- correlate with average user ratings?**

Recipes with many steps might be harder to execute, leading to failures and lower ratings. Alternatively, complex recipes could represent higher-quality dishes that earn higher praise. Understanding this helps both recipe creators and users.

---

### Relevant Columns

| Column | Source | Description |
|---|---|---|
| `n_steps` | `RAW_recipes.csv` | Number of preparation steps |
| `n_ingredients` | computed | Number of ingredients |
| `minutes` | `RAW_recipes.csv` | Estimated preparation time in minutes |
| `calories` | `RAW_recipes.csv` | Caloric content (from `nutrition` column) |
| `avg_rating` | computed | Average rating across all user reviews |

In [2]:
# Load the two raw CSV files
recipes = pd.read_csv(Path('data') / 'RAW_recipes.csv')
interactions = pd.read_csv(Path('data') / 'RAW_interactions.csv')

print(f'Recipes:      {recipes.shape[0]:,} rows x {recipes.shape[1]} columns')
print(
    f'Interactions: {interactions.shape[0]:,} '
    f'rows x {interactions.shape[1]} columns')
recipes.head(3)

Recipes:      83,782 rows x 12 columns
Interactions: 731,927 rows x 5 columns


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,1 brownies in the world best ever,333281,40,985201,2008-10-27,"['60-minutes-or-less', 'time-to-make', 'course...","[138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]",10,['heat the oven to 350f and arrange the rack i...,"these are the most; chocolatey, moist, rich, d...","['bittersweet chocolate', 'unsalted butter', '...",9
1,1 in canada chocolate chip cookies,453467,45,1848091,2011-04-11,"['60-minutes-or-less', 'time-to-make', 'cuisin...","[595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]",12,"['pre-heat oven the 350 degrees f', 'in a mixi...",this is the recipe that we use at my school ca...,"['white sugar', 'brown sugar', 'salt', 'margar...",11
2,412 broccoli casserole,306168,40,50969,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]",6,"['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",9


## Step 2: Data Cleaning and Exploratory Data Analysis

In [3]:
# Data Cleaning

# 1. Replace 0 ratings with NaN.
#    Food.com stores 'no rating given' as 0, not an actual 0-star rating.
interactions['rating'] = interactions['rating'].replace(0, np.nan)

# 2. Compute average rating per recipe across all reviews.
avg_ratings = (
    interactions
    .groupby('recipe_id')['rating']
    .mean()
    .reset_index()
    .rename(columns={'recipe_id': 'id', 'rating': 'avg_rating'})
)

# 3. Left-join so every recipe is kept (recipes with no reviews get NaN).
df = recipes.merge(avg_ratings, on='id', how='left')

# 4. Parse the nutrition column from a string-encoded list into separate
#    columns.
#    Order: [calories, total_fat, sugar, sodium, protein, saturated_fat,
#    carbohydrates]
df['nutrition'] = df['nutrition'].apply(eval)
nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium',
                  'protein', 'saturated_fat', 'carbohydrates']
df[nutrition_cols] = pd.DataFrame(df['nutrition'].tolist(), index=df.index)

# 5. Parse ingredients from string to list, then count them.
df['ingredients'] = df['ingredients'].apply(eval)
df['n_ingredients'] = df['ingredients'].apply(len)

# 6. Convert submission date to datetime.
df['submitted'] = pd.to_datetime(df['submitted'])

# 7. Drop extreme outlier prep times (> 1 day = likely data entry errors).
df = df[df['minutes'] <= 1440].copy()

print(f'Cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
cols = ['name', 'n_steps', 'n_ingredients',
        'minutes', 'calories', 'avg_rating']
df[cols].head()


Cleaned dataset: 83,194 rows x 20 columns


,name,n_steps,n_ingredients,minutes,calories,avg_rating
0,1 brownies in the world best ever,10,9,40,138.4,4.0
1,1 in canada chocolate chip cookies,12,11,45,595.1,5.0
2,412 broccoli casserole,6,9,40,194.8,5.0
3,millionaire pound cake,7,7,120,878.3,5.0
4,2000 meatloaf,17,13,90,267.0,5.0


### Univariate Analysis

In [4]:
# Univariate Plot 1: Distribution of average ratings.
# Ratings are heavily skewed toward 4-5 stars, suggesting a positivity bias on
#    Food.com.
fig = px.histogram(
    df.dropna(subset=['avg_rating']),
    x='avg_rating',
    nbins=20,
    title='Distribution of Average Recipe Ratings',
    labels={'avg_rating': 'Average Rating (Stars)',
            'count': 'Number of Recipes'},
    color_discrete_sequence=['#2196F3']
)
fig.update_layout(
    xaxis_title='Average Rating (Stars)',
    yaxis_title='Number of Recipes',
    bargap=0.05,
    template='plotly_white'
)
fig.show()
fig.write_html('assets/avg-rating-distribution.html', include_plotlyjs='cdn')

In [5]:
# Univariate Plot 2: Distribution of number of steps per recipe.
# Most recipes have fewer than 15 steps; distribution is right-skewed.
fig2 = px.histogram(
    df,
    x='n_steps',
    nbins=40,
    title='Distribution of Number of Steps in Recipes',
    labels={'n_steps': 'Number of Steps', 'count': 'Number of Recipes'},
    color_discrete_sequence=['#FF7043']
)
fig2.update_layout(
    xaxis_title='Number of Steps',
    yaxis_title='Number of Recipes',
    bargap=0.05,
    template='plotly_white'
)
fig2.show()
fig2.write_html('assets/n-steps-distribution.html', include_plotlyjs='cdn')

### Bivariate Analysis

In [6]:
# Bivariate Plot 1: Average rating by number of steps (binned).
# Groups recipes by complexity to see if step count affects ratings.
df['step_bin'] = pd.cut(
    df['n_steps'],
    bins=[0, 5, 10, 15, 20, df['n_steps'].max() + 1],
    labels=['1-5', '6-10', '11-15', '16-20', '20+'],
    right=True
)

fig3 = px.box(
    df.dropna(subset=['avg_rating', 'step_bin']),
    x='step_bin',
    y='avg_rating',
    title='Average Rating by Recipe Complexity (Number of Steps)',
    labels={'step_bin': 'Number of Steps',
            'avg_rating': 'Average Rating (Stars)'},
    color='step_bin',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_layout(
    xaxis_title='Number of Steps (Binned)',
    yaxis_title='Average Rating (Stars)',
    template='plotly_white',
    showlegend=False
)
fig3.show()
fig3.write_html('assets/rating-by-steps.html', include_plotlyjs='cdn')

In [7]:
# Bivariate Plot 2: Average rating vs. number of ingredients (sample of 5,000).
# Checks whether ingredient count -- another complexity proxy -- affects
#    ratings.
fig4 = px.scatter(
    df.dropna(subset=['avg_rating']).sample(5000, random_state=42),
    x='n_ingredients',
    y='avg_rating',
    trendline='lowess',
    title='Average Rating vs. Number of Ingredients (Sample of 5,000 Recipes)',
    labels={
        'n_ingredients': 'Number of Ingredients',
        'avg_rating': 'Average Rating (Stars)'},
    opacity=0.3,
    color_discrete_sequence=['#7E57C2']
)
fig4.update_layout(template='plotly_white')
fig4.show()
fig4.write_html('assets/rating-vs-ingredients.html', include_plotlyjs='cdn')

### Interesting Aggregates

In [8]:
# Grouped aggregate table: recipe statistics by step complexity bin.
agg_table = (
    df.dropna(subset=['avg_rating'])
    .groupby('step_bin', observed=True)
    .agg(
        num_recipes=('id', 'count'),
        mean_rating=('avg_rating', 'mean'),
        mean_calories=('calories', 'mean'),
        mean_n_ingredients=('n_ingredients', 'mean'),
        mean_minutes=('minutes', 'mean')
    )
    .round(2)
)
agg_table

,num_recipes,mean_rating,mean_calories,mean_n_ingredients,mean_minutes
step_bin,,,,,
1-5,18466,4.64,327.67,6.94,44.74
6-10,31744,4.62,400.60,8.79,58.57
11-15,18248,4.62,463.18,10.33,66.75
16-20,7338,4.64,536.80,11.52,79.03
20+,4835,4.65,645.79,12.84,103.19


## Step 3: Assessment of Missingness

### MNAR Analysis

The `avg_rating` column has non-trivial missingness. Recipes that were never reviewed end up with a missing average rating. We think this is likely **NMAR** because the value itself plays a role in whether it gets observed. Recipes that would get low ratings are probably less likely to be reviewed in the first place, since users mostly review recipes they actually made. If a recipe looks unappealing or has a bad description, fewer people will cook it, and fewer people will leave a rating. So the quality of the recipe (which is what the rating measures) also affects whether someone rates it at all. That's the definition of NMAR: the missingness depends on the unobserved value.

One piece of additional data that could help explain the missingness is page views or impressions for each recipe. If we knew how many people saw a recipe but chose not to rate it, we could condition on visibility, and the missingness might become explainable by observed columns (making it MAR instead).

### Missingness Dependency

We look at the missingness of `avg_rating` and test whether it depends on other columns using permutation tests. For quantitative columns, we use the absolute difference in means as our test statistic.

In [9]:
# Check missingness counts across columns
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nTotal rows: {len(df):,}')
print(
    f'avg_rating missing: '
    f'{df["avg_rating"].isna().sum():,} '
    f'({df["avg_rating"].isna().mean():.2%})')


Missing values per column:
name              1
description      69
avg_rating     2563
dtype: int64

Total rows: 83,194
avg_rating missing: 2,563 (3.08%)


#### Test 1: Missingness of `avg_rating` vs. `n_steps` (dependent)

**Null Hypothesis:** The distribution of `n_steps` is the same when `avg_rating` is missing vs. when it is not missing.

**Alternative Hypothesis:** The distributions are different.

**Test Statistic:** Absolute difference in group means.

**Significance Level:** 0.05

In [10]:
# Permutation test: missingness of avg_rating vs. n_steps
df['rating_missing'] = df['avg_rating'].isna()

missing_steps = df.loc[df['rating_missing'], 'n_steps']
not_missing_steps = df.loc[~df['rating_missing'], 'n_steps']

observed_diff_steps = abs(missing_steps.mean() - not_missing_steps.mean())

n_permutations = 1000
perm_diffs_steps = np.array([])
combined_steps = df['n_steps'].values
n_missing = missing_steps.shape[0]

for _ in range(n_permutations):
    shuffled = np.random.permutation(combined_steps)
    perm_diff = abs(shuffled[:n_missing].mean() - shuffled[n_missing:].mean())
    perm_diffs_steps = np.append(perm_diffs_steps, perm_diff)

p_value_steps = np.mean(perm_diffs_steps >= observed_diff_steps)

print(f'Mean n_steps (rating missing):   {missing_steps.mean():.4f}')
print(f'Mean n_steps (rating observed):  {not_missing_steps.mean():.4f}')
print(f'Observed abs diff in means:      {observed_diff_steps:.4f}')
print(f'P-value: {p_value_steps:.4f}')
print()
if p_value_steps < 0.05:
    print(
        'Result: Reject H0. The missingness of '
        'avg_rating DOES depend on n_steps.')
else:
    print('Result: Fail to reject H0. No significant dependency.')

Mean n_steps (rating missing):   11.4663
Mean n_steps (rating observed):  10.0304
Observed abs diff in means:      1.4359
P-value: 0.0000

Result: Reject H0. The missingness of avg_rating DOES depend on n_steps.


In [11]:
# Visualization: empirical distribution of permutation test for n_steps
fig_steps = px.histogram(
    x=perm_diffs_steps,
    nbins=50,
    title='Missingness Permutation Test: avg_rating vs. n_steps',
    labels={'x': 'Absolute Difference in Mean n_steps', 'y': 'Count'},
    color_discrete_sequence=['#42A5F5']
)
fig_steps.add_vline(
    x=observed_diff_steps,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Observed = {observed_diff_steps:.4f}',
    annotation_position='top right'
)
fig_steps.update_layout(
    xaxis_title='Absolute Difference in Mean n_steps',
    yaxis_title='Count',
    template='plotly_white',
    showlegend=False
)
fig_steps.show()
fig_steps.write_html('assets/missingness-steps.html', include_plotlyjs='cdn')

#### Test 2: Missingness of `avg_rating` vs. `calories` (dependent)

**Null Hypothesis:** The distribution of `calories` is the same when `avg_rating` is missing vs. when it is not missing.

**Alternative Hypothesis:** The distributions are different.

**Test Statistic:** Absolute difference in group means.

**Significance Level:** 0.05

In [12]:
# Permutation test: missingness of avg_rating vs. calories
missing_cal = df.loc[df['rating_missing'], 'calories']
not_missing_cal = df.loc[~df['rating_missing'], 'calories']

observed_diff_cal = abs(missing_cal.mean() - not_missing_cal.mean())

n_permutations = 1000
perm_diffs_cal = np.array([])
combined_cal = df['calories'].values
n_missing_cal = missing_cal.shape[0]

for _ in range(n_permutations):
    shuffled = np.random.permutation(combined_cal)
    perm_diff = abs(
        shuffled[:n_missing_cal].mean() - shuffled[n_missing_cal:].mean())
    perm_diffs_cal = np.append(perm_diffs_cal, perm_diff)

p_value_cal = np.mean(perm_diffs_cal >= observed_diff_cal)

print(f'Mean calories (rating missing):   {missing_cal.mean():.4f}')
print(f'Mean calories (rating observed):  {not_missing_cal.mean():.4f}')
print(f'Observed abs diff in means:       {observed_diff_cal:.4f}')
print(f'P-value: {p_value_cal:.4f}')
print()
if p_value_cal < 0.05:
    print(
        'Result: Reject H0. The missingness of '
        'avg_rating DOES depend on calories.')
else:
    print(
        'Result: Fail to reject H0. '
        'The missingness of avg_rating does '
        'NOT depend on calories.')


Mean calories (rating missing):   510.0081
Mean calories (rating observed):  425.1600
Observed abs diff in means:       84.8481
P-value: 0.0000

Result: Reject H0. The missingness of avg_rating DOES depend on calories.


In [13]:
# Visualization: empirical distribution of permutation test for calories
fig_cal = px.histogram(
    x=perm_diffs_cal,
    nbins=50,
    title='Missingness Permutation Test: avg_rating vs. calories',
    labels={'x': 'Absolute Difference in Mean Calories', 'y': 'Count'},
    color_discrete_sequence=['#66BB6A']
)
fig_cal.add_vline(
    x=observed_diff_cal,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Observed = {observed_diff_cal:.2f}',
    annotation_position='top right'
)
fig_cal.update_layout(
    xaxis_title='Absolute Difference in Mean Calories',
    yaxis_title='Count',
    template='plotly_white',
    showlegend=False
)
fig_cal.show()
fig_cal.write_html('assets/missingness-calories.html', include_plotlyjs='cdn')

#### Test 3: Missingness of `avg_rating` vs. `sodium` (not dependent)

**Null Hypothesis:** The distribution of `sodium` is the same when `avg_rating` is missing vs. when it is not missing.

**Alternative Hypothesis:** The distributions are different.

**Test Statistic:** Absolute difference in group means.

**Significance Level:** 0.05

In [14]:
# Permutation test: missingness of avg_rating vs. sodium
missing_sodium = df.loc[df['rating_missing'], 'sodium']
not_missing_sodium = df.loc[~df['rating_missing'], 'sodium']

observed_diff_sodium = abs(missing_sodium.mean() - not_missing_sodium.mean())

n_permutations = 1000
perm_diffs_sodium = np.array([])
combined_sodium = df['sodium'].values
n_missing_sodium = missing_sodium.shape[0]

for _ in range(n_permutations):
    shuffled = np.random.permutation(combined_sodium)
    perm_diff = abs(
        shuffled[:n_missing_sodium].mean()
        - shuffled[n_missing_sodium:].mean())
    perm_diffs_sodium = np.append(perm_diffs_sodium, perm_diff)

p_value_sodium = np.mean(perm_diffs_sodium >= observed_diff_sodium)

print(f'Mean sodium (rating missing):   {missing_sodium.mean():.4f}')
print(f'Mean sodium (rating observed):  {not_missing_sodium.mean():.4f}')
print(f'Observed abs diff in means:     {observed_diff_sodium:.4f}')
print(f'P-value: {p_value_sodium:.4f}')
print()
if p_value_sodium < 0.05:
    print(
        'Result: Reject H0. The missingness '
        'of avg_rating DOES depend on sodium.')
else:
    print(
        'Result: Fail to reject H0. '
        'The missingness of avg_rating does '
        'NOT depend on sodium.')


Mean sodium (rating missing):   27.8474
Mean sodium (rating observed):  28.6216
Observed abs diff in means:     0.7742
P-value: 0.7470

Result: Fail to reject H0. The missingness of avg_rating does NOT depend on sodium.


In [15]:
# Visualization: empirical distribution of permutation test for sodium
fig_sodium = px.histogram(
    x=perm_diffs_sodium,
    nbins=50,
    title='Missingness Permutation Test: avg_rating vs. sodium',
    labels={'x': 'Absolute Difference in Mean Sodium', 'y': 'Count'},
    color_discrete_sequence=['#FFA726']
)
fig_sodium.add_vline(
    x=observed_diff_sodium,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Observed = {observed_diff_sodium:.2f}',
    annotation_position='top right'
)
fig_sodium.update_layout(
    xaxis_title='Absolute Difference in Mean Sodium',
    yaxis_title='Count',
    template='plotly_white',
    showlegend=False
)
fig_sodium.show()
fig_sodium.write_html('assets/missingness-sodium.html', include_plotlyjs='cdn')

In [16]:
# Clean up helper column
df = df.drop(columns=['rating_missing'])

## Step 4: Hypothesis Testing

In [17]:
# Hypothesis Test: Does recipe complexity affect average ratings?
#
# Null Hypothesis (H0):
#   The distribution of average ratings for high-step recipes (above median)
#   is the same as for low-step recipes. Any observed difference is due to
#    chance.
#
# Alternative Hypothesis (Ha):
#   Recipes with more steps (above median) have LOWER average ratings.
#
# Test Statistic: mean(low-step ratings) - mean(high-step ratings)
#   A positive value supports Ha (simpler recipes rated higher).
#
# Significance Level: alpha = 0.05
# Method: Permutation test (no distributional assumptions needed).

median_steps = df['n_steps'].median()
rated = df.dropna(subset=['avg_rating']).copy()

low  = rated[rated['n_steps'] <= median_steps]['avg_rating']
high = rated[rated['n_steps'] >  median_steps]['avg_rating']

observed_diff = low.mean() - high.mean()

#perm test
n_permutations = 1000
diffs = np.array([])

ratings = rated['avg_rating'].values
n_low = low.shape[0]

for _ in range(n_permutations):
    shuffled = np.random.permutation(ratings)
    perm_diff = shuffled[:n_low].mean() - shuffled[n_low:].mean()
    diffs = np.append(diffs, perm_diff)

p_value = np.mean(diffs >= observed_diff)

print(f'Median steps:                    {median_steps}')
print(f'Mean rating -- low-step recipes:  {low.mean():.4f}')
print(f'Mean rating -- high-step recipes: {high.mean():.4f}')
print(f'Observed difference in means:     {observed_diff:.4f}')
print(f'P-value ({n_permutations} permutations):        {p_value:.4f}')
print()
if p_value < 0.05:
    print('Result: Reject the null hypothesis at alpha = 0.05.')
else:
    print('Result: Fail to reject the null hypothesis at alpha = 0.05.')
    print(
        'There is no significant evidence that '
        'complex recipes receive lower ratings.')


Median steps:                    9.0
Mean rating -- low-step recipes:  4.6266
Mean rating -- high-step recipes: 4.6234
Observed difference in means:     0.0032
P-value (1000 permutations):        0.2240

Result: Fail to reject the null hypothesis at alpha = 0.05.
There is no significant evidence that complex recipes receive lower ratings.


## Step 5: Framing a Prediction Problem

In [18]:
# Prediction Problem: Predict the average rating of a recipe (REGRESSION).
#
# Response Variable: avg_rating
#   Chosen because it directly measures how well a recipe is received,
#   which ties to our central question about complexity and quality.
#
# Evaluation Metric: RMSE (Root Mean Squared Error)
#   RMSE penalizes large errors more than MAE, which matters since
#   predicting a 1-star recipe as 5-star is far worse than a 0.1-star error.
#   RMSE is also interpretable in the original star-rating units.
#
# Features (all known BEFORE any ratings are submitted):
#   - n_steps        : number of preparation steps
#   - n_ingredients  : number of ingredients
#   - minutes        : estimated prep time
#   - calories       : caloric content
#   - sugar          : sugar content (% daily value)
#   - total_fat      : fat content (% daily value)
#
# We do NOT use review text or any rating-related data since those only
# exist after users have already rated the recipe.

print('Response variable: avg_rating')
print(f'Recipes with avg_rating: {df["avg_rating"].notna().sum():,}')
print(f'Recipes missing rating:  {df["avg_rating"].isna().sum():,}')
cols = ['n_steps', 'n_ingredients', 'minutes',
        'calories', 'sugar', 'total_fat',
        'avg_rating']
df[cols].dropna().describe().round(2)


Response variable: avg_rating
Recipes with avg_rating: 80,631
Recipes missing rating:  2,563


,n_steps,n_ingredients,minutes,calories,sugar,total_fat,avg_rating
count,80631.00,80631.00,80631.00,80631.00,80631.00,80631.00,80631.00
mean,10.03,9.20,61.79,425.16,66.79,32.29,4.63
std,6.27,3.81,96.80,607.77,214.99,58.64,0.64
min,1.00,1.00,0.00,0.00,0.00,0.00,1.00
25%,6.00,6.00,20.00,171.30,9.00,8.00,4.50
50%,9.00,9.00,35.00,304.40,23.00,20.00,5.00
75%,13.00,12.00,60.00,495.60,60.00,39.00,5.00
max,93.00,37.00,1440.00,45609.00,18127.00,3464.00,5.00


## Step 6: Baseline Model

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

model_df = df[
    ['n_steps', 'n_ingredients', 'minutes',
     'calories', 'step_bin',
     'avg_rating']].dropna()

X = model_df[['n_steps', 'n_ingredients', 'minutes', 'calories', 'step_bin']]
y = model_df['avg_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

quantitative_features = ['n_steps', 'n_ingredients', 'minutes', 'calories']
nominal_features = ['step_bin']

preprocessor = ColumnTransformer(
    transformers=[
        ('quant', 'passthrough', quantitative_features),
        ('nom', OneHotEncoder(drop='first', handle_unknown='ignore'),
         nominal_features)
    ]
)
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

baseline_pipeline.fit(X_train, y_train)

train_rmse = root_mean_squared_error(y_train,
                                     baseline_pipeline.predict(X_train))
test_rmse  = root_mean_squared_error(y_test,
                                     baseline_pipeline.predict(X_test))

print(f'Training RMSE: {train_rmse:.4f}')
print(f'Test RMSE:     {test_rmse:.4f}')
print(f'\nFor reference, avg_rating std dev: {y.std():.4f}')
print(
    f'Baseline (always predict mean) RMSE: '
    f'{((y_test - y_train.mean())**2).mean()**0.5:.4f}')


Training RMSE: 0.6394
Test RMSE:     0.6425

For reference, avg_rating std dev: 0.6403
Baseline (always predict mean) RMSE: 0.6428


## Step 7: Final Model

In [20]:
from sklearn.preprocessing import FunctionTransformer, QuantileTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

model_df = df[['n_steps', 'n_ingredients', 'minutes', 'calories', 'sugar',
               'total_fat', 'protein', 'step_bin', 'avg_rating']].dropna()

X = model_df.drop(columns='avg_rating')
y = model_df['avg_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)


### Feature Engineering Rationale

**New Feature 1: `log_calories`:** Calories are heavily right-skewed with extreme outliers (up to ~45,000). A log transform compresses these outliers and makes the relationship with the response more linear. This is more useful than raw calories because the difference between 200 and 400 calories matters a lot more than the difference between 10,000 and 10,200.

**New Feature 2: `protein_to_fat_ratio`:** The ratio of protein to total fat captures how "healthy" a recipe is. Health-conscious users might rate nutritionally balanced recipes higher. Neither protein nor fat alone tells you this, but together as a ratio they give a useful signal about recipe quality.

**Standardization via `StandardScaler`:** Applied to `n_steps` and `n_ingredients` so the tree-based model can split more effectively on these features relative to the nutritional columns.

**Quantile transformation via `QuantileTransformer`:** Applied to `minutes` to handle its heavy skew and map it to a uniform distribution, which reduces the influence of extreme prep times.

### Hyperparameter Tuning Plan

We tune `max_depth` of a `RandomForestRegressor`. Tree depth controls model complexity directly: too shallow and the model underfits; too deep and it overfits to noise. Since our baseline linear model barely beats the mean-prediction baseline, a non-linear model with controlled depth should be able to capture patterns that linear regression misses.

In [21]:
def add_log_calories(X):
    X = X.copy()
    X['log_calories'] = np.log1p(X['calories'])
    return X

def add_protein_fat_ratio(X):
    X = X.copy()
    X['protein_to_fat_ratio'] = X['protein'] / (X['total_fat'] + 1)
    return X

log_cal_transformer = FunctionTransformer(add_log_calories, validate=False)
pf_ratio_transformer = FunctionTransformer(add_protein_fat_ratio,
                                           validate=False)

final_pipeline = Pipeline([
    ('log_cal', log_cal_transformer),
    ('pf_ratio', pf_ratio_transformer),
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('scale', StandardScaler(), ['n_steps', 'n_ingredients']),
            ('quantile', QuantileTransformer(output_distribution='uniform',
             random_state=42), ['minutes']),
            ('passthrough', 'passthrough', ['sugar', 'log_calories',
             'protein_to_fat_ratio']),
            ('nom', OneHotEncoder(drop='first', handle_unknown='ignore',
             sparse_output=False), ['step_bin'])
        ],
        remainder='drop'
    )),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42,
     n_jobs=-1))
])

param_grid = {'regressor__max_depth': [5, 10, 15, 20, 30, None]}

grid_search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

results = pd.DataFrame(grid_search.cv_results_)[
    ['param_regressor__max_depth', 'mean_test_score', 'std_test_score']]
results.columns = ['max_depth', 'mean_cv_rmse', 'std_cv_rmse']
results['mean_cv_rmse'] = -results['mean_cv_rmse']
results = results.sort_values('mean_cv_rmse')
print('GridSearchCV Results (sorted by RMSE):')
print(results.to_string(index=False))
print(f'\nBest max_depth: {grid_search.best_params_["regressor__max_depth"]}')

GridSearchCV Results (sorted by RMSE):
max_depth  mean_cv_rmse  std_cv_rmse
        5      0.638608     0.007582
       10      0.639458     0.007587
       15      0.642059     0.007670
       20      0.646557     0.007474
       30      0.653931     0.007225
     None      0.656604     0.007449

Best max_depth: 5


In [22]:
best_depth = grid_search.best_params_['regressor__max_depth']
final_model = grid_search.best_estimator_

final_train_rmse = root_mean_squared_error(y_train,
                                           final_model.predict(X_train))
final_test_rmse = root_mean_squared_error(y_test, final_model.predict(X_test))

print(f'Best max_depth: {best_depth}')
print(f'Final Model: Training RMSE: {final_train_rmse:.4f}')
print(f'Final Model: Test RMSE:     {final_test_rmse:.4f}')
print(f'\nBaseline Model: Test RMSE:  0.6425')
print(f'Improvement:                  {0.6425 - final_test_rmse:.4f}')

Best max_depth: 5
Final Model: Training RMSE: 0.6360
Final Model: Test RMSE:     0.6413

Baseline Model: Test RMSE:  0.6425
Improvement:                  0.0012


## Step 8: Fairness Analysis

### Fairness Analysis: Low-Calorie vs. High-Calorie Recipes

**Question:** Does our model perform worse for high-calorie recipes than for low-calorie recipes?

We split recipes into two groups based on the median calorie count in the test set:
- **Group X (low-calorie):** recipes with calories at or below the median
- **Group Y (high-calorie):** recipes with calories above the median

**Evaluation metric:** RMSE, since this is a regression task.

**Null Hypothesis:** Our model is fair; its RMSE for low-calorie and high-calorie recipes is roughly the same, and any difference is due to random chance.

**Alternative Hypothesis:** Our model is unfair; its RMSE for high-calorie recipes is higher than for low-calorie recipes.

**Test statistic:** RMSE(high-calorie) - RMSE(low-calorie). A positive value supports the alternative.

**Significance level:** 0.05

In [23]:
eval_df = X_test.copy()
eval_df['avg_rating'] = y_test.values
eval_df['predicted'] = final_model.predict(X_test)

eval_df['calorie_group'] = np.where(
    eval_df['calories'] <= eval_df['calories'].median(),
    'low-calorie', 'high-calorie'
)

rmse_low = root_mean_squared_error(
    eval_df.loc[eval_df['calorie_group'] == 'low-calorie', 'avg_rating'],
    eval_df.loc[eval_df['calorie_group'] == 'low-calorie', 'predicted']
)
rmse_high = root_mean_squared_error(
    eval_df.loc[eval_df['calorie_group'] == 'high-calorie', 'avg_rating'],
    eval_df.loc[eval_df['calorie_group'] == 'high-calorie', 'predicted']
)

observed_stat = rmse_high - rmse_low

print(f'RMSE for low-calorie recipes:  {rmse_low:.4f}')
print(f'RMSE for high-calorie recipes: {rmse_high:.4f}')
print(f'Observed difference (high - low): {observed_stat:.4f}')

RMSE for low-calorie recipes:  0.6473
RMSE for high-calorie recipes: 0.6353
Observed difference (high - low): -0.0120


In [24]:
actuals = eval_df['avg_rating'].values
predictions = eval_df['predicted'].values
groups = eval_df['calorie_group'].values

n_permutations = 1000
perm_stats = np.array([])

for _ in range(n_permutations):
    shuffled_groups = np.random.permutation(groups)
    high_mask = shuffled_groups == 'high-calorie'
    low_mask = shuffled_groups == 'low-calorie'

    perm_rmse_high = root_mean_squared_error(actuals[high_mask],
                                             predictions[high_mask])
    perm_rmse_low = root_mean_squared_error(actuals[low_mask],
                                            predictions[low_mask])
    perm_stats = np.append(perm_stats, perm_rmse_high - perm_rmse_low)

p_value = np.mean(perm_stats >= observed_stat)

print(f'Observed test statistic: {observed_stat:.4f}')
print(f'P-value ({n_permutations} permutations): {p_value:.4f}')
print()
if p_value < 0.05:
    print('Result: Reject the null hypothesis at alpha = 0.05.')
    print(
        'There is significant evidence that '
        'the model performs worse for '
        'high-calorie recipes.')
else:
    print('Result: Fail to reject the null hypothesis at alpha = 0.05.')
    print(
        'There is no significant evidence of '
        'unfairness between calorie groups.')


Observed test statistic: -0.0120
P-value (1000 permutations): 0.7770

Result: Fail to reject the null hypothesis at alpha = 0.05.
There is no significant evidence of unfairness between calorie groups.


In [25]:
fig = px.histogram(
    x=perm_stats,
    nbins=50,
    title='Fairness Permutation Test: RMSE(High-Calorie) - RMSE(Low-Calorie)',
    labels={'x': 'RMSE Difference', 'y': 'Count'},
    color_discrete_sequence=['#42A5F5']
)
fig.add_vline(
    x=observed_stat,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Observed = {observed_stat:.4f}',
    annotation_position='top right'
)
fig.update_layout(
    xaxis_title='RMSE Difference (High - Low Calorie)',
    yaxis_title='Count',
    template='plotly_white',
    showlegend=False
)
fig.show()

fig.write_html('assets/fairness-analysis.html', include_plotlyjs='cdn')